# MENTORX — Knowledge tracing on FoundationalASSIST (deep learning track)

**Question:** given a student's earlier answers, how well can we predict whether they answer the
*next* question correctly on the first try, and which model should MENTORX use for that?

**Dataset:** [FoundationalASSIST](https://huggingface.co/datasets/ASSISTments/FoundationalASSIST)
(Worden, Heffernan, Heffernan & Sonkar, AIED 2026, arXiv:2602.00070). It has 5,000 students,
~1.72M first attempts, 3,395 Illustrative Mathematics grade 6–8 problems with full text and answer
keys, and 224 Common Core skills. Licence: **CC BY-NC 4.0**, gated on Hugging Face, for research
and education only. Cite the paper, and do not redistribute the raw CSVs in the repo.

**Label:** `discrete_score = 1` only when the first attempt was correct **with no hint and the answer
not shown**. So the *current* row's `hint_count` / `saw_answer` reveal the label and must never be
model inputs. The same columns from *earlier* rows are fair, useful history.

**Models compared (same student split, same metrics):**

| # | Model | Kind |
|---|---|---|
| 1 | Skill success rate | baseline |
| 2 | Question success rate | baseline (item difficulty) |
| 3 | Gradient boosting on history features | classical ML (what MENTORX serves today) |
| 4 | DKT — LSTM over (skill, response) | deep learning, original notebook's model |
| 5 | **DKT-Q** — LSTM + question embeddings + per-question difficulty + hint/time history | deep learning, improved |
| 6 | **Transformer-KT** — causal self-attention version of model 5 (SAKT-style) | deep learning, improved |
| 7 | Ensemble of the best deep model and gradient boosting | stacking |

**Protocol:** split **by student** 70/15/15 (seed 42). All tuning, early stopping and model choice use
validation students only. Test students are scored once, at the end.

### How to run this on Kaggle
1. **Settings → Accelerator → GPU T4 x1** (on CPU this takes several hours).
2. Attach the dataset `piyushgoyal1021/foundational-assist-dataset`. The paths below are the same ones the
   first notebook used: `/kaggle/input/datasets/piyushgoyal1021/foundational-assist-dataset/`.
3. **Run All.** Outputs (models, charts, `results.json`, `train_log.txt`) go to `/kaggle/working/mentorx_kt/`.
4. The **last cell** prints one summary block. Copy that block, or download the whole notebook
   (File → Download notebook), and send it back.

In [ ]:
import copy
import gc
import html
import json
import math
import os
import re
import time
from decimal import Decimal, InvalidOperation
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, train_test_split

SEED = 42


def set_seed(seed: int = SEED) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)


# Same dataset path as the first notebook
BASE_PATH = "/kaggle/input/datasets/piyushgoyal1021/foundational-assist-dataset/"
DATA_DIR = Path(BASE_PATH)
if not (DATA_DIR / "Interactions.csv").exists():
    # Kaggle sometimes mounts datasets one level up; find Interactions.csv anywhere under /kaggle/input
    found = sorted(Path("/kaggle/input").rglob("Interactions.csv"))
    if not found:
        raise FileNotFoundError("Attach the foundational-assist-dataset to this notebook (Add data).")
    DATA_DIR = found[0].parent

OUT_DIR = Path("/kaggle/working/mentorx_kt")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Training budget. Early stopping usually ends runs well before MAX_EPOCHS.
MAX_EPOCHS = 30
if DEVICE.type == "cpu":
    print("WARNING: no GPU. Turn on Settings -> Accelerator -> GPU T4, or this will take several hours.")
PATIENCE = 3
BATCH_SIZE = 64

print("Data:", DATA_DIR)
print("Outputs:", OUT_DIR.resolve())
print("Device:", DEVICE, "| torch", torch.__version__, "| MAX_EPOCHS", MAX_EPOCHS)

## 1. Load and clean

Findings from the audit, applied here in one place:
- 31,441 rows repeat an interaction `id` with identical content, so they are exported twice. Keep one.
- 27 rows repeat a `problem_id` in Problems (one has a different answer-key format). Keep the first.
- 8,952 rows have no score. Drop them, since there is nothing to learn from.
- `end_time` mixes `.873+00` and `.06+00` formats. Without `format="ISO8601"`, pandas silently
  turned ~1,750 valid timestamps into NaT, and the old notebook dropped those rows. Only the 3,445
  truly missing timestamps are dropped now.
- Questions tagged with 2–3 skills (≈13% of attempts) are **kept**. The old notebook dropped them,
  which removed data and left holes in every student's sequence. Each row now carries up to 3 skills.

In [ ]:
raw = pd.read_csv(DATA_DIR / "Interactions.csv", index_col=0)
problems_raw = pd.read_csv(DATA_DIR / "Problems.csv")
skills_raw = pd.read_csv(DATA_DIR / "Skills.csv")

df = raw.drop_duplicates(subset="id").dropna(subset=["discrete_score"]).copy()
df["end_time"] = pd.to_datetime(df["end_time"], utc=True, format="ISO8601", errors="coerce")
df = df.dropna(subset=["end_time"])
df["correct"] = df["discrete_score"].astype(np.int8)
df = df.sort_values(["user_id", "end_time", "id"]).reset_index(drop=True)

problems = problems_raw.drop_duplicates(subset="problem_id").reset_index(drop=True)
skills = skills_raw.drop_duplicates(subset=["problem_id", "skill_id"])

print(f"Interactions: {len(raw):,} raw -> {len(df):,} clean")
del raw
df = df.drop(columns=["answer_text", "discrete_score"])  # not used below; saves memory
gc.collect()
print(f"Students {df.user_id.nunique():,} | questions {df.problem_id.nunique():,} | "
      f"skills {skills.skill_id.nunique()} | correct rate {df.correct.mean():.3f}")
print("Attempts per student:", df.groupby("user_id").size().describe()[["min", "50%", "max"]].to_dict())
print("Skills per question:", skills.groupby("problem_id").size().value_counts().sort_index().to_dict())

## 2. Index skills and questions, split by student

- Skill and question indices are metadata only, with no outcomes, so they can be built from the full tables.
- A question that **no training student attempted** maps to index 0, which acts as "unknown". The model
  therefore never sees a learned embedding for a question it couldn't have learned from.

In [ ]:
student_ids = np.sort(df["user_id"].unique())
train_students, temp = train_test_split(student_ids, test_size=0.30, random_state=SEED)
val_students, test_students = train_test_split(temp, test_size=0.50, random_state=SEED)
split_of = pd.Series("train", index=student_ids)
split_of[val_students] = "val"
split_of[test_students] = "test"
df["split"] = df["user_id"].map(split_of).values

# Skills: index 0 is padding, real skills are 1..S
skill_ids = np.sort(skills["skill_id"].unique())
skill_idx = {s: i + 1 for i, s in enumerate(skill_ids)}
NUM_SKILLS = len(skill_ids) + 1
MAX_SK = 3
prob_skills = skills.groupby("problem_id")["skill_id"].apply(lambda s: [skill_idx[x] for x in s][:MAX_SK])
sk = np.zeros((len(df), MAX_SK), dtype=np.int64)
for j, lst in enumerate(df["problem_id"].map(prob_skills)):
    sk[j, : len(lst)] = lst
df["n_skills"] = (sk > 0).sum(1)
df["skill1"] = sk[:, 0]  # primary skill, used by the feature-based models

# Questions: index 0 = unknown (not attempted by any training student)
train_q = np.sort(df.loc[df.split == "train", "problem_id"].unique())
q_idx = {q: i + 1 for i, q in enumerate(train_q)}
NUM_Q = len(train_q) + 1
df["q"] = df["problem_id"].map(q_idx).fillna(0).astype(np.int64)

for name in ["train", "val", "test"]:
    part = df[df.split == name]
    print(f"{name:5s}: {part.user_id.nunique():5,} students  {len(part):9,} interactions  "
          f"unknown-question rows {(part.q == 0).mean():.2%}")

## 3. Evaluation helpers

Metrics: **AUC** (ranking), **log loss** (probability quality, used to choose models), accuracy, Brier
score and **ECE** (calibration). MENTORX acts on the probabilities directly, for example by targeting
P(correct) ≈ 0.65, so calibration matters as much as AUC. Test confidence intervals come from
resampling whole students (bootstrap).

In [ ]:
def ece_score(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    total = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            total += m.mean() * abs(y[m].mean() - p[m].mean())
    return total


def metrics(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        "AUC": roc_auc_score(y, p),
        "LogLoss": log_loss(y, p),
        "Accuracy": accuracy_score(y, p >= 0.5),
        "Brier": brier_score_loss(y, p),
        "ECE": ece_score(y, p),
    }


def bootstrap_auc_ci(y, p, groups, reps=200, seed=SEED):
    rng = np.random.default_rng(seed)
    uniq, inv = np.unique(groups, return_inverse=True)
    rows_of = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    aucs = []
    for _ in range(reps):
        pick = rng.integers(0, len(uniq), len(uniq))
        rows = np.concatenate([rows_of[g] for g in pick])
        aucs.append(roc_auc_score(y[rows], p[rows]))
    return np.percentile(aucs, [2.5, 97.5])


VAL_PRED, TEST_PRED = {}, {}  # model name -> predictions aligned with the val/test rows below
val_rows = df.index[df.split == "val"].to_numpy()
test_rows = df.index[df.split == "test"].to_numpy()
y_val = df.loc[val_rows, "correct"].to_numpy()
y_test = df.loc[test_rows, "correct"].to_numpy()

## 4. Baselines: skill and question success rates (training students only)

In [ ]:
tr = df[df.split == "train"]
glob = tr.correct.mean()
# Smoothed rates: (correct + 2*glob) / (attempts + 2) so rare skills/questions shrink to the average
def smoothed_rate(key):
    g = tr.groupby(key)["correct"].agg(["sum", "count"])
    return (g["sum"] + 2 * glob) / (g["count"] + 2)


skill_rate = smoothed_rate("skill1")
q_rate = smoothed_rate("problem_id")
for name, rows in [("val", val_rows), ("test", test_rows)]:
    part = df.loc[rows]
    store = VAL_PRED if name == "val" else TEST_PRED
    store["Skill rate"] = part["skill1"].map(skill_rate).fillna(glob).to_numpy()
    store["Question rate"] = part["problem_id"].map(q_rate).fillna(glob).to_numpy()

display(pd.DataFrame({m: metrics(y_val, VAL_PRED[m]) for m in VAL_PRED}).T.round(4))

## 5. Gradient boosting on leakage-free history features

This is the same idea as MENTORX's deployed model (`ml/mentorx_ml/features.py`): every feature uses only
the student's **earlier** rows, plus question/skill difficulty learned from **training students**.
For training rows, the difficulty is *cross-fitted* over 5 student folds, so a row's own label never feeds
its own feature. New here: the previous attempt's hints / answer-reveal and the time gaps.

In [ ]:
def history_features(d: pd.DataFrame) -> pd.DataFrame:
    d = d.sort_values(["user_id", "end_time", "id"])
    f = pd.DataFrame(index=d.index)
    u = d.groupby("user_id", sort=False)
    c = d["correct"].astype(float)
    n_prev = u.cumcount()
    cum = u["correct"].cumsum() - c  # correct answers before this row
    f["log_prior_overall"] = np.log1p(n_prev)
    f["acc_overall"] = (cum + 1) / (n_prev + 2)
    cs = u["correct"].cumsum()
    last10 = cs.groupby(d["user_id"]).shift(1).fillna(0) - cs.groupby(d["user_id"]).shift(11).fillna(0)
    f["acc_last10"] = (last10 + 1) / (np.minimum(n_prev, 10) + 2)
    f["prev_correct"] = u["correct"].shift(1).fillna(-1)
    f["prev_hints"] = np.log1p(u["hint_count"].shift(1).fillna(0))
    f["prev_saw_answer"] = u["saw_answer"].shift(1).astype(float).fillna(0)
    hint_cum = u["hint_count"].transform(lambda s: (s > 0).cumsum()) - (d["hint_count"] > 0)
    f["hint_rate_overall"] = (hint_cum + 0.5) / (n_prev + 1)
    secs = d["end_time"].astype("int64") / 1e9
    f["log_secs_since_any"] = np.log1p(secs - secs.groupby(d["user_id"]).shift(1)).fillna(-1)

    k = d.groupby(["user_id", "skill1"], sort=False)
    n_sk = k.cumcount()
    cum_sk = k["correct"].cumsum() - c
    f["log_prior_skill"] = np.log1p(n_sk)
    f["acc_skill"] = (cum_sk + 1) / (n_sk + 2)
    cs_sk = k["correct"].cumsum()
    keys = [d["user_id"], d["skill1"]]
    last3 = cs_sk.groupby(keys).shift(1).fillna(0) - cs_sk.groupby(keys).shift(4).fillna(0)
    f["acc_skill_last3"] = (last3 + 1) / (np.minimum(n_sk, 3) + 2)
    f["prev_skill_correct"] = k["correct"].shift(1).fillna(-1)
    pos = n_prev.astype(float)
    f["log_gap_skill"] = np.log1p(pos - pos.groupby(keys).shift(1)).fillna(-1)
    f["log_secs_since_skill"] = np.log1p(secs - secs.groupby(keys).shift(1)).fillna(-1)
    f["n_skills"] = d["n_skills"]
    return f.loc[d.index]


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def difficulty(fit: pd.DataFrame, apply: pd.DataFrame, g: float) -> pd.DataFrame:
    out = pd.DataFrame(index=apply.index)
    for key, name in [("problem_id", "item"), ("skill1", "skill")]:
        s = fit.groupby(key)["correct"].agg(["sum", "count"])
        rate = (s["sum"] + 5 * g) / (s["count"] + 5)
        out[f"{name}_logit"] = logit(apply[key].map(rate).fillna(g).to_numpy())
        out[f"log_{name}_count"] = np.log1p(apply[key].map(s["count"]).fillna(0).to_numpy())
    return out


t0 = time.time()
H = history_features(df)
D = pd.DataFrame(index=df.index, columns=["item_logit", "log_item_count", "skill_logit", "log_skill_count"], dtype=float)
tr_idx = df.index[df.split == "train"]
folds = GroupKFold(n_splits=5).split(tr_idx, groups=df.loc[tr_idx, "user_id"])
for fit_pos, pred_pos in folds:
    D.loc[tr_idx[pred_pos]] = difficulty(df.loc[tr_idx[fit_pos]], df.loc[tr_idx[pred_pos]], glob).values
other = df.index[df.split != "train"]
D.loc[other] = difficulty(df.loc[tr_idx], df.loc[other], glob).values
X = pd.concat([H, D.astype(float)], axis=1)
print("Features:", list(X.columns), f"({time.time() - t0:.0f}s)")

gbm = HistGradientBoostingClassifier(
    learning_rate=0.1, max_leaf_nodes=63, max_iter=400, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=20, random_state=0,
)
gbm.fit(X.loc[tr_idx], df.loc[tr_idx, "correct"])
VAL_PRED["Gradient boosting"] = gbm.predict_proba(X.loc[val_rows])[:, 1]
TEST_PRED["Gradient boosting"] = gbm.predict_proba(X.loc[test_rows])[:, 1]
del H, D, X  # free ~0.5 GB before deep-model training (the run is sized for an 8 GB laptop)
gc.collect()
print(f"Trees: {gbm.n_iter_} ({time.time() - t0:.0f}s)")
display(pd.DataFrame({m: metrics(y_val, VAL_PRED[m]) for m in VAL_PRED}).T.round(4))

## 6. Sequence tensors for the deep models

One sequence per student, in time order. Position *t* holds interaction *t*. The model reads
interactions 0..t and predicts interaction *t+1*, so the inputs at *t* are only:
- the question and skills of attempt *t*, and whether it was correct;
- the hints / answer-reveal on attempt *t*, which are known once *t* is finished;
- log seconds since the attempt before *t*.

The target side is the question and skills of attempt *t+1*, with **no** hint, time or label information.

In [ ]:
d_sorted = df  # already sorted by user, time, id
secs_all = d_sorted["end_time"].astype("int64").to_numpy() / 1e9
users = d_sorted["user_id"].to_numpy()
new_user = np.r_[True, users[1:] != users[:-1]]
gap = np.r_[0.0, np.diff(secs_all)]
gap[new_user] = 0.0
feat_all = np.stack([
    np.log1p(d_sorted["hint_count"].clip(upper=10).to_numpy()),
    d_sorted["saw_answer"].astype(float).to_numpy(),
    np.log1p(np.maximum(gap, 0)) / 10.0,
], axis=1).astype(np.float32)


USER_START = np.flatnonzero(new_user)
USER_END = np.r_[USER_START[1:], len(d_sorted)]
Q_ALL = np.array(d_sorted["q"].to_numpy(), copy=True)
R_ALL = np.array(d_sorted["correct"].to_numpy(), dtype=np.int64, copy=True)
SPLIT_ALL = d_sorted["split"].to_numpy()


def build_sequences(split: str):
    seqs = []
    for a, b in zip(USER_START, USER_END):  # each student's rows are one contiguous block
        if SPLIT_ALL[a] != split:
            continue
        seqs.append({"rows": np.arange(a, b), "q": Q_ALL[a:b], "sk": sk[a:b], "r": R_ALL[a:b], "f": feat_all[a:b]})
    return seqs


SEQ = {s: build_sequences(s) for s in ["train", "val", "test"]}
MAX_LEN = max(len(x["r"]) for v in SEQ.values() for x in v)
print({k: len(v) for k, v in SEQ.items()}, "| longest sequence:", MAX_LEN)


def collate(batch):
    L = max(len(x["r"]) for x in batch) - 1
    B = len(batch)
    out = {
        "q_in": torch.zeros(B, L, dtype=torch.long), "sk_in": torch.zeros(B, L, MAX_SK, dtype=torch.long),
        "r_in": torch.zeros(B, L, dtype=torch.long), "f_in": torch.zeros(B, L, 3),
        "q_out": torch.zeros(B, L, dtype=torch.long), "sk_out": torch.zeros(B, L, MAX_SK, dtype=torch.long),
        "y": torch.zeros(B, L), "mask": torch.zeros(B, L, dtype=torch.bool), "lengths": torch.zeros(B, dtype=torch.long),
    }
    for i, x in enumerate(batch):
        n = len(x["r"]) - 1
        out["q_in"][i, :n] = torch.from_numpy(x["q"][:-1])
        out["sk_in"][i, :n] = torch.from_numpy(x["sk"][:-1])
        out["r_in"][i, :n] = torch.from_numpy(x["r"][:-1])
        out["f_in"][i, :n] = torch.from_numpy(x["f"][:-1])
        out["q_out"][i, :n] = torch.from_numpy(x["q"][1:])
        out["sk_out"][i, :n] = torch.from_numpy(x["sk"][1:])
        out["y"][i, :n] = torch.from_numpy(x["r"][1:].astype(np.float32))
        out["mask"][i, :n] = True
        out["lengths"][i] = n
    return out


def loader(split, shuffle):
    return torch.utils.data.DataLoader(SEQ[split], batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate)

## 7. Models

- **DKT**: the original notebook's model, an LSTM over (skill × response) embeddings. Multi-skill questions
  average their skills' embeddings.
- **DKT-Q** adds what DKT is missing: a **question embedding** (same skill, very different questions), a
  learned **per-question difficulty bias** (a Rasch/IRT-style term, which on ASSISTments data typically adds
  several AUC points), and the previous attempt's hints / answer-reveal / time gap.
- **Transformer-KT** uses the same inputs and head as DKT-Q, with the LSTM replaced by a 2-layer causal
  self-attention encoder (learned positions, 4 heads, feed-forward blocks). It is a stronger version of
  the old single-layer SAKT, which had no feed-forward block and no question information.

In [ ]:
class SkillBag(nn.Module):
    """Mean of up to 3 skill embeddings (index 0 = padding)."""

    def __init__(self, n, dim):
        super().__init__()
        self.emb = nn.Embedding(n, dim, padding_idx=0)

    def forward(self, idx):  # (B, L, 3) -> (B, L, dim)
        e = self.emb(idx)
        cnt = (idx > 0).sum(-1, keepdim=True).clamp(min=1)
        return e.sum(-2) / cnt


class KTModel(nn.Module):
    def __init__(self, encoder="lstm", use_question=True, dim=128, dropout=0.2, layers=2, heads=4):
        super().__init__()
        self.use_question = use_question
        self.encoder_type = encoder
        self.skill_resp = SkillBag(2 * NUM_SKILLS, dim)  # (skill, response) like classic DKT
        self.skill_out = SkillBag(NUM_SKILLS, dim)
        in_dim = dim
        head_in = dim + dim
        if use_question:
            self.q_resp = nn.Embedding(2 * NUM_Q, dim, padding_idx=0)
            self.q_out = nn.Embedding(NUM_Q, dim, padding_idx=0)
            self.q_bias = nn.Embedding(NUM_Q, 1, padding_idx=0)
            nn.init.zeros_(self.q_bias.weight)
            in_dim = dim + dim + 3
            head_in = dim + dim + dim
        self.inp = nn.Sequential(nn.Linear(in_dim, dim), nn.LayerNorm(dim), nn.Dropout(dropout))
        if encoder == "lstm":
            self.enc = nn.LSTM(dim, dim, batch_first=True, num_layers=1)
        else:
            self.pos = nn.Embedding(MAX_LEN + 1, dim)
            layer = nn.TransformerEncoderLayer(dim, heads, 4 * dim, dropout, batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(head_in, dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim, 1))

    def encode(self, b):
        sk_in, r_in = b["sk_in"], b["r_in"]
        sr = torch.where(sk_in > 0, 2 * sk_in + r_in.unsqueeze(-1), torch.zeros_like(sk_in))
        parts = [self.skill_resp(sr)]
        if self.use_question:
            qr = torch.where(b["q_in"] > 0, 2 * b["q_in"] + r_in, torch.zeros_like(r_in))
            parts += [self.q_resp(qr), b["f_in"]]
        x = self.inp(torch.cat(parts, -1))
        L = x.size(1)
        if self.encoder_type == "lstm":
            packed = nn.utils.rnn.pack_padded_sequence(x, b["lengths"].cpu(), batch_first=True, enforce_sorted=False)
            h, _ = self.enc(packed)
            h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=L)
        else:
            x = x + self.pos(torch.arange(L, device=x.device)).unsqueeze(0)
            causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), 1)
            h = self.enc(x, mask=causal, src_key_padding_mask=~b["mask"])
        return h

    def predict_from_state(self, h, q_out, sk_out):
        parts = [h, self.skill_out(sk_out)]
        if self.use_question:
            parts.append(self.q_out(q_out))
        logit = self.head(torch.cat(parts, -1)).squeeze(-1)
        if self.use_question:
            logit = logit + self.q_bias(q_out).squeeze(-1)
        return logit

    def forward(self, b):
        return self.predict_from_state(self.encode(b), b["q_out"], b["sk_out"])


def to_device(b):
    return {k: v.to(DEVICE) for k, v in b.items()}


@torch.no_grad()
def predict(model, split):
    """Predictions aligned with df rows of that split (the first attempt of each student has none)."""
    model.eval()
    probs = np.full(len(df), np.nan)
    for batch_seqs in [SEQ[split][i:i + BATCH_SIZE] for i in range(0, len(SEQ[split]), BATCH_SIZE)]:
        b = to_device(collate(batch_seqs))
        p = torch.sigmoid(model(b)).cpu().numpy()
        for i, x in enumerate(batch_seqs):
            probs[x["rows"][1:]] = p[i, : len(x["r"]) - 1]
    return probs


def train_model(name, model, lr=1e-3, weight_decay=1e-5):
    set_seed()
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=1)
    bce = nn.BCEWithLogitsLoss(reduction="sum")
    best, best_state, bad, hist = math.inf, None, 0, []
    vr = np.flatnonzero(df.split.to_numpy() == "val")
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name}: {n_params:,} parameters")
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0, tot, cnt = time.time(), 0.0, 0
        for b in loader("train", True):
            b = to_device(b)
            logits = model(b)
            loss = bce(logits[b["mask"]], b["y"][b["mask"]]) / b["mask"].sum()
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item() * b["mask"].sum().item()
            cnt += b["mask"].sum().item()
        p = predict(model, "val")[vr]
        keep = ~np.isnan(p)
        m = metrics(df["correct"].to_numpy()[vr][keep], p[keep])
        sched.step(m["LogLoss"])
        hist.append({"epoch": epoch, "train_loss": tot / cnt, **{f"val_{k}": v for k, v in m.items()}})
        line = (f"{name} epoch {epoch:2d} | train {tot / cnt:.4f} | val logloss {m['LogLoss']:.4f} "
                f"AUC {m['AUC']:.4f} acc {m['Accuracy']:.4f} | lr {opt.param_groups[0]['lr']:.0e} | {time.time() - t0:.0f}s")
        print(line)
        with open(OUT_DIR / "train_log.txt", "a") as fh:  # progress you can watch while the notebook runs
            fh.write(line + "\n")
        if m["LogLoss"] < best - 1e-4:
            best, best_state, bad = m["LogLoss"], copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stop")
                break
    model.load_state_dict(best_state)
    torch.save({"state_dict": best_state, "num_skills": NUM_SKILLS, "num_q": NUM_Q, "max_len": MAX_LEN},
               ckpt_path(name))
    pd.DataFrame(hist).to_csv(ckpt_path(name).with_suffix(".history.csv"), index=False)
    return model, pd.DataFrame(hist)


def ckpt_path(name):
    return OUT_DIR / f"{name.lower().replace(' ', '_').replace('-', '_')}.pt"


def train_or_load(name, model):
    """Resume support: reuse a finished model from OUT_DIR instead of retraining it.
    Delete the .pt file (or set RETRAIN=1) to force training."""
    path = ckpt_path(name)
    if path.exists() and os.environ.get("RETRAIN") != "1":
        ckpt = torch.load(path, map_location=DEVICE)
        assert (ckpt["num_skills"], ckpt["num_q"]) == (NUM_SKILLS, NUM_Q), "checkpoint is from different data"
        model.load_state_dict(ckpt["state_dict"])
        hist_file = path.with_suffix(".history.csv")
        hist = pd.read_csv(hist_file) if hist_file.exists() else pd.DataFrame()
        print(f"{name}: loaded finished model from {path.name} (training history: {len(hist)} epochs)")
        return model.to(DEVICE), hist
    return train_model(name, model)


def first_attempt_fill(p, rows):
    """Each student's very first attempt has no history; fall back to the question rate there."""
    out = p[rows]
    miss = np.isnan(out)
    out[miss] = df.loc[rows[miss], "problem_id"].map(q_rate).fillna(glob).to_numpy()
    return out

In [ ]:
MODELS, HISTORIES = {}, {}
for name, kwargs in [
    ("DKT", dict(encoder="lstm", use_question=False)),
    ("DKT-Q", dict(encoder="lstm", use_question=True)),
    ("Transformer-KT", dict(encoder="transformer", use_question=True)),
]:
    model, hist = train_or_load(name, KTModel(**kwargs))
    MODELS[name], HISTORIES[name] = model, hist
    VAL_PRED[name] = first_attempt_fill(predict(model, "val"), val_rows)
    gc.collect()

display(pd.DataFrame({m: metrics(y_val, VAL_PRED[m]) for m in VAL_PRED}).T.round(4))

### Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in HISTORIES.items():
    axes[0].plot(h["epoch"], h["val_AUC"], marker="o", label=name)
    axes[1].plot(h["epoch"], h["val_LogLoss"], marker="o", label=name)
axes[0].set(title="Validation AUC", xlabel="epoch")
axes[1].set(title="Validation log loss", xlabel="epoch")
axes[0].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "learning_curves.png", dpi=120)
plt.show()

## 8. Ensemble (chosen on validation)

The deep model learns sequence patterns, while gradient boosting uses explicit counts and time features.
Averaging their **logits** often helps. The weight is chosen on validation students only.

In [ ]:
best_dl = min(["DKT", "DKT-Q", "Transformer-KT"], key=lambda m: metrics(y_val, VAL_PRED[m])["LogLoss"])
weights = np.linspace(0, 1, 21)
scores = [log_loss(y_val, 1 / (1 + np.exp(-(w * logit(VAL_PRED[best_dl]) + (1 - w) * logit(VAL_PRED["Gradient boosting"])))))
          for w in weights]
W = float(weights[int(np.argmin(scores))])
ens_name = f"Ensemble ({best_dl} + GBM)"
VAL_PRED[ens_name] = 1 / (1 + np.exp(-(W * logit(VAL_PRED[best_dl]) + (1 - W) * logit(VAL_PRED["Gradient boosting"]))))
print(f"Best deep model on validation: {best_dl}; ensemble weight on it: {W:.2f}")

val_table = pd.DataFrame({m: metrics(y_val, VAL_PRED[m]) for m in VAL_PRED}).T.sort_values("LogLoss")
display(val_table.round(4))

## 9. Final test (run once, after every choice above was fixed on validation)

In [ ]:
for name in ["DKT", "DKT-Q", "Transformer-KT"]:
    TEST_PRED[name] = first_attempt_fill(predict(MODELS[name], "test"), test_rows)
TEST_PRED[ens_name] = 1 / (1 + np.exp(-(W * logit(TEST_PRED[best_dl]) + (1 - W) * logit(TEST_PRED["Gradient boosting"]))))

groups = df.loc[test_rows, "user_id"].to_numpy()
rows_out = []
for name in val_table.index:
    m = metrics(y_test, TEST_PRED[name])
    lo, hi = bootstrap_auc_ci(y_test, TEST_PRED[name], groups, reps=200)
    rows_out.append({"Model": name, **m, "AUC 95% CI": f"{lo:.3f}–{hi:.3f}"})
test_table = pd.DataFrame(rows_out).set_index("Model")
test_table.to_csv(OUT_DIR / "test_metrics.csv")
json.dump({"val": val_table.round(5).to_dict(orient="index"), "test": test_table.round(5).to_dict(orient="index"),
           "ensemble_weight": W, "best_deep_model": best_dl, "seed": SEED},
          open(OUT_DIR / "results.json", "w"), indent=2, default=str)
display(test_table.round(4))

### Calibration (test students)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for name in ["Gradient boosting", best_dl, ens_name]:
    p = TEST_PRED[name]
    bins = np.clip((p * 10).astype(int), 0, 9)
    xs = [p[bins == b].mean() for b in range(10) if (bins == b).sum() > 100]
    ys = [y_test[bins == b].mean() for b in range(10) if (bins == b).sum() > 100]
    ax.plot(xs, ys, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="predicted P(correct)", ylabel="observed correct rate", title="Reliability (test)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "reliability_test.png", dpi=120)
plt.show()

## 10. Using the model: mastery estimate and next-question choice

Fixes compared with the old prototype:
- **One pass per student.** The history is encoded once, then *all* candidates are scored from that state.
  The old code ran the full LSTM 194 times.
- **Question-level probabilities.** DKT-Q predicts P(correct) for each specific question, so the next
  question is the unseen one whose prediction is closest to MENTORX's target of **0.65**. This replaces two
  rules that disagreed: population success-rate buckets and the "two right answers → harder" override.
- **No answer leak.** `Fill-in Options` equals the answer key for 97% of fill-in questions, so it is never shown.
  Students see the question text and, for multiple choice, the options.

In [ ]:
TARGET_P = 0.65
SERVE = MODELS["DKT-Q"]
skill_name = skills.drop_duplicates("skill_id").set_index("skill_id")[["node_code", "node_name"]]
q_text = problems.set_index("problem_id")["Problem Body"].map(
    lambda v: re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html.unescape(str(v)))).strip())
bank = problems[problems.problem_id.isin(q_idx)].copy()
bank["q"] = bank.problem_id.map(q_idx)
bank_sk = np.zeros((len(bank), MAX_SK), dtype=np.int64)
for j, lst in enumerate(bank.problem_id.map(prob_skills)):
    bank_sk[j, : len(lst)] = lst


@torch.no_grad()
def student_state(history_rows):
    """Encode a student's history (df row indices, time-ordered) and return the last hidden state."""
    x = {"q": df.loc[history_rows, "q"].to_numpy(), "sk": sk[history_rows],
         "r": df.loc[history_rows, "correct"].to_numpy().astype(np.int64), "f": feat_all[history_rows]}
    # collate() drops the last element as "target", so append a dummy target
    x = {k: np.concatenate([v, v[-1:]]) for k, v in x.items()}
    b = to_device(collate([x]))
    SERVE.eval()
    h = SERVE.encode(b)
    return h[0, -1]


@torch.no_grad()
def score_questions(h, q, sk_rows):
    n = len(q)
    hh = h.unsqueeze(0).unsqueeze(0).expand(1, n, -1)
    q_t = torch.as_tensor(q, device=DEVICE).unsqueeze(0)
    s_t = torch.as_tensor(sk_rows, device=DEVICE).unsqueeze(0)
    return torch.sigmoid(SERVE.predict_from_state(hh, q_t, s_t))[0].cpu().numpy()


def recommend_next(history_rows, n=5):
    h = student_state(history_rows)
    seen = set(df.loc[history_rows, "problem_id"])
    cand = bank[~bank.problem_id.isin(seen)].copy()
    cand["p_correct"] = score_questions(h, cand["q"].to_numpy(), bank_sk[~bank.problem_id.isin(seen).to_numpy()])
    cand["distance"] = (cand["p_correct"] - TARGET_P).abs()
    cand["question"] = cand.problem_id.map(q_text).str.slice(0, 120)
    show = ["problem_id", "Problem Type", "question", "Multiple Choice Options", "p_correct"]  # never answer keys
    return cand.sort_values("distance").head(n)[show]


def skill_mastery(history_rows):
    """Average predicted P(correct) over each skill's questions = a per-skill mastery estimate."""
    h = student_state(history_rows)
    p = score_questions(h, bank["q"].to_numpy(), bank_sk)
    per_skill = pd.DataFrame({"skill_idx": bank_sk[:, 0], "p": p}).groupby("skill_idx")["p"].mean()
    out = per_skill.rename("mastery").to_frame()
    out["skill_id"] = [skill_ids[i - 1] for i in out.index]
    return out.join(skill_name, on="skill_id").sort_values("mastery")


demo_student = SEQ["test"][0]
hist_rows = demo_student["rows"][: int(len(demo_student["rows"]) * 0.7)]
print("History length:", len(hist_rows))
display(skill_mastery(hist_rows).head(8).round(3))
display(recommend_next(hist_rows).round(3))

### Is "closest to 0.65" actually calibrated for the questions it picks?

This checks the policy on test students without any live data. For every test attempt the model gave
0.55–0.75, what fraction were actually answered correctly?

In [ ]:
p = TEST_PRED["DKT-Q"]
band = (p >= 0.55) & (p <= 0.75)
print(f"Test attempts predicted in [0.55, 0.75]: {band.sum():,}; mean predicted {p[band].mean():.3f}; "
      f"observed correct {y_test[band].mean():.3f}")

## 11. Answer checking (fixed)

Bugs fixed from the prototype:
- `"1,000"` used to be split on the comma into `1` and `000`, so an answer of **1** was marked correct.
  Several accepted answers are separated by **comma + space** (`"30.68, 30.69"`), while thousands use a bare
  comma, and the split now follows that rule.
- Multiple-choice options are compared after HTML is stripped, and "Check All That Apply" needs the exact set.
- There is one `check_answer`, not two definitions that drift apart.

In [ ]:
def _clean(v):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html.unescape(str(v)))).strip().casefold()


def _num(v):
    try:
        return Decimal(str(v).strip().replace(",", ""))
    except InvalidOperation:
        return None


PROB = problems.set_index("problem_id")


def check_answer(problem_id, submitted):
    if problem_id not in PROB.index:
        return {"status": "error", "message": "Question not found."}
    q = PROB.loc[problem_id]
    kind = str(q["Answer Types"]).strip()
    if kind == "Numeric":
        accepted = [a for a in (_num(x) for x in re.split(r",\s+", str(q["Fill-in Answers"]))) if a is not None]
        got = _num(submitted)
        if got is None:
            return {"status": "invalid", "message": "Enter a number."}
        ok = got in accepted
    elif kind == "Multiple Choice":
        ok = _clean(submitted) == _clean(q["Multiple Choice Answers"])
    elif kind == "Check All That Apply":
        want = {_clean(x) for x in str(q["Multiple Choice Answers"]).split("||") if _clean(x)}
        items = submitted.split("||") if isinstance(submitted, str) else list(submitted)
        ok = {_clean(x) for x in items if _clean(x)} == want
    else:
        return {"status": "unsupported", "message": f"Answer type not supported yet: {kind}"}
    return {"status": "evaluated", "is_correct": bool(ok)}


# Real tests (they fail loudly if a fix regresses)
thousands = PROB[(PROB["Answer Types"] == "Numeric") & PROB["Fill-in Answers"].astype(str).str.fullmatch(r"1,000")].index[0]
assert check_answer(thousands, "1000")["is_correct"]
assert check_answer(thousands, "1,000")["is_correct"]
assert not check_answer(thousands, "1")["is_correct"], "thousands separator bug is back"
assert check_answer(7411, "30.68")["is_correct"] and check_answer(7411, "30.69")["is_correct"]
assert not check_answer(7411, "30.7")["is_correct"]
assert check_answer(178083, "About 2,700 in²")["is_correct"]
assert not check_answer(178083, "About 900 in²")["is_correct"]
full = ["Radius and diameter of a circle", "Radius and circumference of a circle", "Diameter and circumference of a circle"]
assert check_answer(172163, full)["is_correct"]
assert not check_answer(172163, full[:2])["is_correct"]
print("All answer-checking tests passed.")

## 12. Results summary and next steps

The table in §9 holds the real test numbers from this run. In the report, quote them from
`results.json`, never typed by hand.

**What changed versus the first notebook**
1. Data: valid timestamps no longer dropped (format fix); multi-skill questions kept; duplicate cells removed.
2. Leakage rules written down: the current row's hints/answer-reveal decide the label, so they are only used
   from *earlier* rows.
3. Models: question embeddings + per-question difficulty + hint/time history (DKT-Q); a 2-layer
   transformer instead of the 1-layer SAKT; gradient boosting and an ensemble as strong non-DL references.
4. Training: fixed seed, AdamW + learning-rate decay on plateau + early stopping in one loop. Before, there
   were repeated manual "extra epoch" cells, and re-running a cell silently kept training the old model.
5. Evaluation: validation-only model choice, one test pass, AUC confidence intervals by student, calibration.
6. Prototype: one encoder pass per student, question-level choice at P≈0.65, no answer-key leak, fixed
   numeric answer checking.

**Next steps**
- Copy the chosen configuration into `ml/` as a script (`ml/scripts/train_dl.py`) and a config file, like the
  existing ASSISTments/EdNet experiments, so `docs/ml-report.md` can cite generated numbers.
- Add the dataset to `docs/datasets.md` (licence CC BY-NC 4.0, gated access, citation).
- Ablations for the report: DKT-Q without question embeddings, without hint/time features, without
  multi-skill rows. Each one shows *why* the accuracy improved.
- Transfer to MENTORX: these models learn IDs of *maths* questions, so they cannot score MENTORX's ML questions
  directly. Either keep the concept-agnostic gradient-boosting model for serving (current design) and present
  deep KT as the research comparison, or fine-tune DKT-Q on pilot data once enough MENTORX answers exist.
- Optional, heavier and novel: embed `Problem Body` text with a sentence-transformer and feed it in place of, or
  alongside, the question ID. That lets the model score **unseen** questions, which is exactly MENTORX's situation.

## 13. Summary to share

Copy everything this cell prints and send it back. It contains every number needed for the report.

In [ ]:
import platform
import sklearn

print("=" * 30, "MENTORX KT RUN SUMMARY", "=" * 30)
print(f"device={DEVICE} gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'} "
      f"torch={torch.__version__} pandas={pd.__version__} sklearn={sklearn.__version__} python={platform.python_version()}")
print(f"rows={len(df):,} students={df.user_id.nunique():,} MAX_EPOCHS={MAX_EPOCHS} seed={SEED} "
      f"best_deep={best_dl} ensemble_weight={W:.2f}")
print("\n--- training log (every epoch) ---")
print(open(OUT_DIR / "train_log.txt").read())
print("--- validation ---")
print(val_table.round(4).to_string())
print("\n--- test (scored once) ---")
print(test_table.round(4).to_string())
band = (TEST_PRED["DKT-Q"] >= 0.55) & (TEST_PRED["DKT-Q"] <= 0.75)
print(f"\nDKT-Q band 0.55-0.75: n={band.sum():,} predicted={TEST_PRED['DKT-Q'][band].mean():.3f} "
      f"observed={y_test[band].mean():.3f}")
print("=" * 84)